# CLIP Training Comparison on Flickr30k

This notebook compares 5 different methods for training/evaluating CLIP on the Flickr30k dataset:
1. **Baseline**: Zero-shot evaluation of the pre-trained model.
2. **Linear Probe**: Training only the projection heads while keeping the backbones frozen.
3. **Partial Fine-tune**: Unfreezing and training only the last few layers of the encoders.
4. **LoRA**: Using Low-Rank Adaptation (PEFT) to fine-tune the model efficiently.
5. **Full Fine-tune**: Unfreezing the entire model for comprehensive training.

At the end, all models are evaluated on Recall@1, Recall@5, and MRR, and then uploaded to Hugging Face.

In [5]:
!pip install -q transformers peft "datasets<3.0.0" huggingface_hub torch torchvision tqdm pillow pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 21.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.


In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import CLIPProcessor, CLIPModel, get_scheduler
from torch.optim import AdamW
from datasets import load_dataset
from tqdm.auto import tqdm
import numpy as np
import json
import os
from PIL import Image
from huggingface_hub import notebook_login, HfApi

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


## 1. Dataset Preparation

We use the `nlphuji/flickr30k` dataset. For training CLIP, we use the standard image-text pairs.

In [2]:
print("Loading Flickr30K dataset...")
dataset = load_dataset("nlphuji/flickr30k", split="test", trust_remote_code=True)
print(f"Dataset size: {len(dataset)}")

class Flickr30kDataset(Dataset):
    def __init__(self, dataset, processor):
        self.dataset = dataset
        self.processor = processor
        self.captions = []
        self.image_indices = []

        for i in range(len(dataset)):
            caps = dataset[i]["caption"]
            if isinstance(caps, str): caps = json.loads(caps)
            for cap in caps:
                self.captions.append(cap)
                self.image_indices.append(i)

    def __len__(self):
        return len(self.captions)

    def __getitem__(self, idx):
        image_idx = self.image_indices[idx]
        image = self.dataset[image_idx]["image"].convert("RGB")
        caption = self.captions[idx]

        inputs = self.processor(text=[caption], images=image, return_tensors="pt", padding="max_length", truncation=True)
        inputs = {k: v.squeeze(0) for k, v in inputs.items()}
        return inputs, image_idx

model_id = "openai/clip-vit-base-patch32"
processor = CLIPProcessor.from_pretrained(model_id)
train_ds = Flickr30kDataset(dataset, processor)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)

Loading Flickr30K dataset...


Generating test split:   0%|          | 0/31014 [00:00<?, ? examples/s]

Dataset size: 31014


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


...

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

## 2. Evaluation Logic

Standard Image-Text Retrieval metrics: Recall@1, Recall@5, and MRR.

In [7]:
def evaluate(model, dataset, processor, device, batch_size=64):
    model.eval()
    all_image_embeds = []
    all_text_embeds = []

    num_images = len(dataset)
    captions = []
    caption_to_image_idx = []

    print("Encoding images...")
    for i in tqdm(range(0, num_images, batch_size)):
        batch_end = min(i + batch_size, num_images)
        images = [dataset[j]["image"].convert("RGB") for j in range(i, batch_end)]
        inputs = processor(images=images, return_tensors="pt", padding=True).to(device)

        with torch.no_grad():
            # Get the output object
            out = model.get_image_features(**inputs)

            # Check if out is a tensor; if not, extract the correct attribute
            # This handles the 'BaseModelOutputWithPooling' error
            embeds = out if isinstance(out, torch.Tensor) else getattr(out, "image_embeds", getattr(out, "pooler_output", out))

            # Now normalization will work
            embeds = embeds / embeds.norm(dim=-1, keepdim=True)
            all_image_embeds.append(embeds.float().cpu())

    # Map captions to images for retrieval evaluation
    for i in range(num_images):
        caps = dataset[i]["caption"]
        if isinstance(caps, str):
            caps = json.loads(caps)
        for cap in caps:
            captions.append(cap)
            caption_to_image_idx.append(i)

    num_captions = len(captions)
    print("Encoding captions...")
    for i in tqdm(range(0, num_captions, batch_size)):
        batch_end = min(i + batch_size, num_captions)
        texts = captions[i:batch_end]
        inputs = processor(text=texts, return_tensors="pt", padding=True, truncation=True).to(device)

        with torch.no_grad():
            # Apply the same extraction logic for text features
            out = model.get_text_features(**inputs)
            embeds = out if isinstance(out, torch.Tensor) else getattr(out, "text_embeds", getattr(out, "pooler_output", out))

            embeds = embeds / embeds.norm(dim=-1, keepdim=True)
            all_text_embeds.append(embeds.float().cpu())

    image_embeds = torch.cat(all_image_embeds, dim=0)
    text_embeds = torch.cat(all_text_embeds, dim=0)

    # Compute similarity scores
    sims = text_embeds @ image_embeds.T
    targets = torch.tensor(caption_to_image_idx)

    # Calculate retrieval ranks
    ranks = []
    for i in range(len(sims)):
        row_sims = sims[i]
        target = targets[i]
        sorted_indices = torch.argsort(row_sims, descending=True)
        rank = (sorted_indices == target).nonzero(as_tuple=True)[0].item() + 1
        ranks.append(rank)

    ranks = np.array(ranks)
    r1 = (ranks <= 1).mean() * 100
    r5 = (ranks <= 5).mean() * 100
    mrr = (1.0 / ranks).mean()

    return {"Recall@1": round(r1, 2), "Recall@5": round(r5, 2), "MRR": round(mrr, 4)}

## 3. Training Function

A simple contrastive training loop using CLIP's built-in loss.

In [4]:
def train_model(model, train_loader, epochs=1, lr=5e-5):
    model.to(device)
    optimizer = AdamW(model.parameters(), lr=lr)
    num_training_steps = epochs * len(train_loader)
    lr_scheduler = get_scheduler(
        name="linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps
    )

    model.train()
    for epoch in range(epochs):
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}")
        for batch, _ in progress_bar:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch, return_loss=True)
            loss = outputs.loss

            loss.backward()
            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()

            progress_bar.set_postfix({"loss": loss.item()})
    return model

## 4. Methods Comparison

We will run each method and log the results.

In [5]:
results_table = {}
def log_result(name, metrics):
    results_table[name] = metrics
    print(f"--- {name} ---")
    for k, v in metrics.items():
        print(f"{k}: {v}")

### 4.2 Linear Probe
Freezing the encoders and training only the projection layers.

In [6]:
model = CLIPModel.from_pretrained(model_id)
for param in model.vision_model.parameters(): param.requires_grad = False
for param in model.text_model.parameters(): param.requires_grad = False

model = train_model(model, train_loader, epochs=1)

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Epoch 1:   0%|          | 0/4846 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Encoding images...


  0%|          | 0/485 [00:00<?, ?it/s]

AttributeError: 'BaseModelOutputWithPooling' object has no attribute 'norm'

In [8]:
metrics = evaluate(model, dataset, processor, device)
log_result("Linear Probe", metrics)

Encoding images...


  0%|          | 0/485 [00:00<?, ?it/s]

Encoding captions...


  0%|          | 0/2423 [00:00<?, ?it/s]

--- Linear Probe ---
Recall@1: 28.28
Recall@5: 51.67
MRR: 0.3953


### 4.3 Partial Fine-tune
Unfreezing the projection layers and the last Transformer block of each encoder.

In [ ]:
model = CLIPModel.from_pretrained(model_id)
for param in model.parameters(): param.requires_grad = False

# Unfreeze projections
for param in model.visual_projection.parameters(): param.requires_grad = True
for param in model.text_projection.parameters(): param.requires_grad = True

# Unfreeze last layer of vision model
for param in model.vision_model.encoder.layers[-1].parameters(): param.requires_grad = True
# Unfreeze last layer of text model
for param in model.text_model.encoder.layers[-1].parameters(): param.requires_grad = True

model = train_model(model, train_loader, epochs=1)
metrics = evaluate(model, dataset, processor, device)
log_result("Partial Fine-tune", metrics)

### 4.4 LoRA (Low-Rank Adaptation)
Using PEFT to add trainable adapters to the attention layers.

In [ ]:
from peft import LoraConfig, get_peft_model

model = CLIPModel.from_pretrained(model_id)
config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
)
model = get_peft_model(model, config)
model.print_trainable_parameters()

model = train_model(model, train_loader, epochs=1)
metrics = evaluate(model, dataset, processor, device)
log_result("LoRA", metrics)

### 4.5 Full Fine-tune
Training all parameters of the model.

In [ ]:
model = CLIPModel.from_pretrained(model_id)
model = train_model(model, train_loader, epochs=1, lr=5e-6) # Lower learning rate for stability
metrics = evaluate(model, dataset, processor, device)
log_result("Full Fine-tune", metrics)

## 5. Final Results Summary

In [ ]:
import pandas as pd
df = pd.DataFrame(results_table).T
print("Final Comparison Table:")
display(df)

## 6. Hugging Face Upload

Log in to your Hugging Face account and upload the best performing model.

In [ ]:
from huggingface_hub import notebook_login
# notebook_login() # Uncomment to login via token

def upload_to_hf(model, repo_name):
    try:
        model.push_to_hub(repo_name)
        print(f"Successfully uploaded to https://huggingface.co/{repo_name}")
    except Exception as e:
        print(f"Upload failed: {e}")

# upload_to_hf(model, "your-username/clip-flickr30k-finetuned")